# 80 · Published injured-PT programs along the package pseudospace (AKI vs control)

The AKI screens fail calibration (0.120 in notebook 71, 0.144 on notebook 76's universe), so no AKI call list can
carry new pathway claims. This notebook is the hypothesis-driven alternative: it takes injured-PT programs published
after ischaemia–reperfusion, fixed before any look at our data, and asks three questions of each.

| Part | Question |
|---|---|
| (a) Replication | Does our IR cohort reproduce the published claim? |
| (b) Localization | Where along the PT is the IR − control difference: which segment, uniform or segment-specific, replicated across the four IR × control pairs? |
| (c) Added information | What does that add to the published claim? |

**Programs** (`docs/results/known-programs/programs_aki.csv`, frozen, sha256 asserted):
- **Confirmatory family I:**
  - Kirita 2020's five injured-PT states (repairing, injured S1/2, injured S3, severe, failed repair), with
    healthy-segment markers removed;
  - the core IRI response of Zhou 2023, up and down (listed in at least 3 of 6 IRI groups);
  - human AKI from Hinze 2022, up and down.
- **Secondary:**
  - Gerhardt 2023's segment-resolved repaired-vs-control lists;
  - Dixon 2022's regional Visium lists;
  - other sources of the same states.

**Protocol.** `docs/results/pt-known-programs.md` (commit b90a682), frozen before this notebook ran, with the
coordinator's decisions O1–O8.

**Inherited bounds.**
- Notebook 72's C14 is not met, so localization is at segment level only.
- Notebook 76: the package drops the most injured units (c15) and labels an injured-PT cluster cTAL (c19). Injured
  programs are therefore attenuated, most in S3. The c19 arm adds those units back.
- The IRI time point is unknown.

**Cohort.** Two control and two IRI male mice. Specimens are the replicates, and everything is descriptive.

**The one refit.** The segment F-test on notebook 76's universe U76, with the same code as notebook 75. 75 fitted it
on 71's universe, which lacks Havcr1, Lcn2, Mki67 and Top2a.

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import sys
import time
from pathlib import Path

N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '24'))
for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, str(N_JOBS))                       # thread cap; never in a cache key
NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
package_aki = results_root / 'package_pseudospace' / 'aki'
run_aki = results_root / 'package_pseudospace' / 'runs' / 'aki'
EXPORT = package_aki / 'pt_units.h5ad'
upstream13 = results_root / 'minimal_pt_scfates'
T71 = results_root / 'pt_pathways_package_pseudospace'            # read only (71's AKI α for the 75 guard)
T75 = results_root / 'pt_package_pathway_methods'
T76 = results_root / 'pt_package_aki'
PROGRAM_DIR = project / 'docs' / 'results' / 'known-programs'      # committed, public-literature derived
output = results_root / 'pt_known_injury_programs'
tables, figures, cache = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures, cache):
    folder.mkdir(parents=True, exist_ok=True)
DS = 'aki_vs_control'
PARTITION_FILES = {'condition': 'species', 'relabeling 1': 'swap_IR2A2_Ctrl1A2', 'relabeling 2': 'swap_IR2A2_Ctrl1A4'}
PROGRAM_SHA = {'programs_aki.csv': '1d01fe15a61f7333983c680e3c7cded1aa3ee4f5ab4cdcf1e7445f06a9949d40',
               'programs_aki_members.csv': 'e00750ad066a8a76b3c870efb6a4c96e56bc7163aa7830ca1bd0ed6df52c667d'}
REQUIRED = [EXPORT, upstream13 / 'ortholog_map_used.csv', run_aki / '02_qc' / 'units_qc.h5ad',
            *[PROGRAM_DIR / f for f in PROGRAM_SHA],
            *[T76 / 'tables' / f for f in ('universe.csv', 'gene_sets.json', 'gene_statistics.csv', 'segment_units.csv',
                                           'placement_units.csv', 'inputs.csv',
                                           *[f'pathway_tests_{v}.csv' for v in ('condition', 'relabeling_1', 'relabeling_2')])],
            *[T75 / 'tables' / f for f in (f'gene_statistics_segment_{DS}.csv', f'deseq2_genes_{DS}.csv')],
            *[data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json' for lib in ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70, 71, 75 and 76 first)')

NOTEBOOK_LOGIC_VERSION = '80.known_programs.1'          # Bump when a cached calculation changes.
FINGERPRINT = 'bb4a80c1b484'                            # confirmed AKI pass-1 labels (notebook 70)
EXTERNAL_COMMIT = '59da04b'
ALPHA, CAMERA_COR, CALIBRATION_MAX = .05, .01, .075
EFFECT_MIN, RATIO_MAX = .25, .25
DF1, DF2 = 6, 12
MIN_MEMBERS, N_BINS, MAX_ROWS = 5, 20, 40
FLOOR = .5
SEGMENTS = ('S1', 'S2', 'S3')
SCOPES = {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3'], 'PT': None, 'S1-S2': ['S1', 'S2']}
PARTS = ['condition', 'relabeling 1', 'relabeling 2']
CONTROLS, IRS = ['Ctrl1A2', 'Ctrl1A4'], ['IR2A2', 'IR2A4']
NAMES = CONTROLS + IRS
CASE, REF = ('IR2A2', 'IR2A4'), ('Ctrl1A2', 'Ctrl1A4')       # relabeling 1 = (c1, r1), relabeling 2 = (c1, r2)
PAIRS = [f'{c}−{r}' for c in CASE for r in REF]
LABEL_ORIENTATION = {'Cyp2e1', 'Cyp7b1', 'Gatm', 'Slc7a13', 'Slc13a3', 'Slc22a6', 'Slc5a2', 'Slc22a7', 'Slc5a12',
                     'Aqp2', 'Aqp4', 'Atp6v0d2', 'Rhcg', 'Slc14a2', 'Wnt7b'}
KIRITA_TIME = {'K-REP': 'from 2 d', 'K-INJ12': '4–12 h', 'K-INJ3': '4–12 h', 'K-SEV': 'acute (with arrest)',
               'K-FR': '14 d to 6 wk'}                    # Kirita 2020's text (protocol 1.5); wording only

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
CONTROL_COLOR, IR_COLOR, RELABEL_COLOR = '#0072B2', '#CC3311', '#9a9a9a'
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 90, 'display.max_rows', 120)
READING, GUARDS = [], []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION)
READING.append(f"Logic version {NOTEBOOK_LOGIC_VERSION!r}, by the coordinator's phase-2 instruction (the protocol named "
               "'80.known_injury_programs.1').")

## 2 · Inputs and guards

U76 is built exactly as notebook 76 built it: two `aki_inputs` calls, the pool (the one-to-one ortholog panel) then
detection with the 13 `mt-` genes out. Guards (not cached):
- **G1:** the export contract, package commit and the confirmed pass-1 fingerprint.
- **G2:** U76 (10,701 genes) and the analysed units (14,048) equal notebook 76's; segment medians are ordered.
- **G3:** 76's cached condition model reproduces its T_spatial and moderated-F z; cameraPR on 76's z reproduces its
  set tests.
- **G4** (section 4): on 71's AKI genes, the U76 segment-F-test refit reproduces 75's T_step, Δ_S and SE wherever
  76's α equals 71's.
- **G5:** the program files match their frozen sha256; members lie in U76; none is a label or orientation gene.
- **G6:** the control segment cuts reproduce 76's segment units.

In [ ]:
import h5py
from anndata.io import read_elem
from scipy import sparse

from pseudospace.aki_along_pt import segment_cuts, segment_units
from pseudospace.discrete_comparators import segment_f_test, swap_rule
from pseudospace.package_inputs import aki_inputs, read_pt_units
from pseudospace.pathway_pipelines import camera_pr, common_grid, moderated_f, partition_groups
from pseudospace.stage_cache import cached_payload, digest
from pseudospace import discrete_comparators, nb_gam
from pseudospace import program_localization as pl

t0 = time.perf_counter()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MOUSE_ORTHO = set(orthologs.mouse_symbol.astype(str))
MT = sorted(g for g in MOUSE_ORTHO if g.startswith('mt-'))
assert len(MT) == 13, MT
UNITS = read_pt_units(EXPORT, 'aki')                                                       # G1
PROV = UNITS.uns['provenance']
assert str(PROV['notebook_logic_version']).startswith('70.'), PROV['notebook_logic_version']
assert str(PROV['package']['commit']).startswith(EXTERNAL_COMMIT), PROV['package']
assert PROV['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT, PROV['fingerprints']['pass1']
panels = PROV['config']['marker_panels_yaml']
label_orientation = ({str(g) for s in ('PT-S1', 'PT-S2', 'PT-S3') for g in panels['label_panels'][s]}
                     | {str(g) for side in ('early', 'late') for g in panels['axis_anchors'][side]})
assert label_orientation == LABEL_ORIENTATION, label_orientation ^ LABEL_ORIENTATION
PANEL = [g for g in UNITS.var_names.astype(str) if g in MOUSE_ORTHO]
U_MT = aki_inputs(UNITS, data_root, orthologs, PANEL)['genes']                 # detected on the pool, mt kept
INP = aki_inputs(UNITS, data_root, orthologs, U_MT, exclude_genes=MT)
GENES = list(INP['genes'])
SETS = INP['gene_sets']
U76 = pd.read_csv(T76 / 'tables' / 'universe.csv').gene.astype(str).tolist()
assert GENES == U76, 'G2: the universe differs from notebook 76'
assert SETS == json.loads((T76 / 'tables' / 'gene_sets.json').read_text()), 'G2: the gene sets differ from notebook 76'
POSITION = np.asarray(INP['position'], float)
SPECIMEN = INP['specimen'].astype(str)
SEGMENT = pd.Series(INP['segment']).str.replace('PT-', '', regex=False).to_numpy()
COUNTS = INP['counts'].tocsr()
LIBRARY = np.asarray(INP['library'], float)
LOG_MEAN_COUNT = np.log(np.asarray(COUNTS.mean(axis=0)).ravel())
assert len(POSITION) == 14048, len(POSITION)
medians = pd.DataFrame({'p': POSITION, 's': SPECIMEN, 'g': SEGMENT}).groupby(['s', 'g']).p.median().unstack()
assert (medians['S1'] < medians['S2']).all() and (medians['S2'] < medians['S3']).all(), medians
PARTITIONS = partition_groups(SPECIMEN, INP['case'])
assert list(PARTITIONS) == ['species', 'swap:IR2A2+Ctrl1A2', 'swap:IR2A2+Ctrl1A4'], list(PARTITIONS)
GUARDS.append(f'G1/G2: export contract; package {EXTERNAL_COMMIT}; pass-1 fingerprint {FINGERPRINT}; U76 ({len(GENES):,} genes), '
              f'sets ({len(SETS):,}) and units ({len(POSITION):,}) equal notebook 76; segment medians ordered')


def stored(folder, stage):
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


STAT76 = pd.read_csv(T76 / 'tables' / 'gene_statistics.csv', header=[0, 1], index_col=0)
assert STAT76.index.astype(str).tolist() == GENES
MODELS = {part: stored(T76, f'gene_model_primary_{f}') for part, f in PARTITION_FILES.items()}
np.testing.assert_allclose(MODELS['condition']['stat|T_spatial'], STAT76['condition'].T_spatial.to_numpy(), rtol=1e-12, atol=0)
again = moderated_f(STAT76['condition'].T_spatial, DF1, STAT76['condition'].LR_spec.to_numpy(), DF2,
                    covariate=LOG_MEAN_COUNT, robust=False)
np.testing.assert_allclose(again.z.to_numpy(), STAT76['condition'].z.to_numpy(), rtol=1e-9, atol=1e-9)
GRID = MODELS['condition']['grid']
np.testing.assert_allclose(GRID, common_grid(POSITION, SPECIMEN, 101), rtol=0, atol=1e-12)
Z_POS = {part: STAT76[part].z.astype(float) for part in PARTS}
TESTS76 = {part: pd.read_csv(T76 / 'tables' / f'pathway_tests_{part.replace(" ", "_")}.csv').set_index('pathway_id') for part in PARTS}
GUARDS.append("G3: notebook 76's cached condition model reproduces its T_spatial and moderated-F z")

PROGRAMS_ALL, META = pl.read_programs(PROGRAM_DIR / 'programs_aki.csv', GENES, LABEL_ORIENTATION,
                                      sha256=PROGRAM_SHA['programs_aki.csv'])
assert sha256(PROGRAM_DIR / 'programs_aki_members.csv') == PROGRAM_SHA['programs_aki_members.csv']
MEMBERS_TABLE = pd.read_csv(PROGRAM_DIR / 'programs_aki_members.csv')
assert all(MEMBERS_TABLE[MEMBERS_TABLE.program_id.eq(p)].gene.tolist() == v for p, v in PROGRAMS_ALL.items())
GUARDS.append(f'G5: program files match their frozen sha256; {len(PROGRAMS_ALL)} programs, every member in U76, '
              'no label/orientation gene')

control = np.isin(SPECIMEN, CONTROLS)
T12, T23 = segment_cuts(POSITION[control], 'PT-' + pd.Series(SEGMENT[control]).astype(str))
S_GRID = segment_units(GRID, T12, T23)
units76 = pd.read_csv(T76 / 'tables' / 'segment_units.csv')
np.testing.assert_allclose(units76['segment units'].to_numpy(), S_GRID, rtol=0, atol=1e-9)        # G6
GUARDS.append(f"G6: control cuts t12 {T12:.3f}, t23 {T23:.3f} reproduce notebook 76's segment units")
print('\n'.join('· ' + g for g in GUARDS))
print(f'Inputs {time.perf_counter() - t0:.0f} s · {len(POSITION):,} units · {len(GENES):,} genes · {len(SETS):,} sets')

## 3 · The frozen programs

Confirmatory family I has nine programs; the secondary programs carry nominal p and are never pooled into the
family. Every member is a one-to-one ortholog in U76, and none is a label or orientation gene (protocol R1–R5).

In [ ]:
META = META.copy()
CONFIRMATORY = [p for p in META.index if META.loc[p, 'role'] == 'confirmatory']
SECONDARY = [p for p in META.index if META.loc[p, 'role'] == 'secondary']
assert len(CONFIRMATORY) == 9, CONFIRMATORY
CLAIM_SIGN = {p: (1 if str(META.loc[p, 'direction']).startswith('up') else -1) for p in META.index}
SCOPE_OF = {p: str(META.loc[p, 'scope']) for p in META.index}
display(META[['role', 'source', 'direction', 'segment', 'n_listed', 'n_non_one_to_one', 'n_unmapped_symbol',
              'n_not_in_universe', 'n_label_orientation_excluded', 'n_members']])

## 4 · Per-gene statistics

**Coordinate-free (new, protocol 3.2–3.3).**
- Pseudobulks: log2((Σ counts + 0.5) / Σ mt-free library) per specimen × scope (S1, S2, S3, PT, S1–S2).
- Contrasts: the IR − control group effect, the 4 IR × control pairs, and the relabeled effects, each centred by its
  median over U76.
- The moderated level t.

**The segment F-test on U76** (the one refit). This is notebook 75's `segment_f_test`, unchanged:
- α is 76's condition α;
- squeezeVar with the log mean count;
- cached.

G4 compares its T_step, Δ_S and SE with 75's on the genes the two universes share.

**Reused.**
- The positional F z and curves of notebook 76.
- The DESeq2 Wald stat of notebook 75, on 71's universe (the conventional comparator).

In [ ]:
def pseudobulks(counts, library, specimen, segment):
    return pl.segment_pseudobulk(counts, library, specimen, segment, SCOPES, genes=GENES, floor=FLOOR)


def level_statistics(rates, covariate):
    stats, pairs = {}, {}
    for scope in rates:
        for part in PARTS:
            eff = pl.partition_effects(rates[scope], CASE, REF, part)
            stats[(scope, part)] = pl.moderated_t(eff.effect, eff.s2, float(eff.df.iloc[0]), covariate=covariate)
        pairs[scope] = pd.DataFrame(pl.pair_contrasts(rates[scope], CASE, REF))
    return stats, pairs


t0 = time.perf_counter()
RATES = pseudobulks(COUNTS, LIBRARY, SPECIMEN, SEGMENT)
LEVEL, PAIR_CON = level_statistics(RATES, LOG_MEAN_COUNT)

code_key = digest([NOTEBOOK_LOGIC_VERSION] + [Path(m.__file__) for m in (discrete_comparators, nb_gam)])
ALPHA76 = {'alpha': MODELS['condition']['alpha'], 'converged': MODELS['condition']['alpha_converged'].astype(bool)}
input_key = {'counts': digest(COUNTS), 'library': LIBRARY, 'specimen': SPECIMEN, 'segment': SEGMENT, 'genes': GENES}
SEGMENT_COLUMNS = ['s2', 's2_prior', 'd0', 's2_post', 'F', 'df_total', 'log_p', 'p', 'z', 'q', 'T_step', 'LR_spec', 'df2',
                   'converged', 'separated', 'delta_S1', 'delta_S2', 'delta_S3', 'se_S1', 'se_S2', 'se_S3']
SEG = {}
for (label, (group, nuisance)), part in zip(PARTITIONS.items(), PARTS):
    payload = cached_payload(
        f"segment_ftest_u76_{label.replace(':', '_').replace('+', '_')}",
        lambda group=group, nuisance=nuisance: {c: v.to_numpy() for c, v in segment_f_test(
            COUNTS, LIBRARY, SEGMENT, group, SPECIMEN, ALPHA76, genes=GENES, nuisance=nuisance,
            covariate=LOG_MEAN_COUNT, robust=False, n_jobs=N_JOBS).items()},
        root=cache, params={'label': label, 'robust': False, 'covariate': 'log mean count'},
        inputs={**input_key, 'group': group, 'nuisance': nuisance, 'alpha': ALPHA76['alpha'],
                'alpha_converged': ALPHA76['converged']}, code=code_key)
    frame = pd.DataFrame({c: payload[c] for c in SEGMENT_COLUMNS}, index=pd.Index(GENES, name='gene'))
    frame[['converged', 'separated']] = frame[['converged', 'separated']].astype(bool)
    assert (frame.df2 == (4 if part == 'condition' else 2)).all() and np.isfinite(frame.z).all()
    SEG[part] = frame
Z_SEG = {part: SEG[part].z.astype(float) for part in PARTS}
pd.concat(SEG, axis=1).to_csv(tables / 'gene_statistics_segment_u76.csv')

# G4: 75's segment F-test on 71's AKI universe, where 76's alpha equals 71's
SEG75 = pd.read_csv(T75 / 'tables' / f'gene_statistics_segment_{DS}.csv', header=[0, 1], index_col=0)
alpha71 = pd.Series(stored(T71, f'gene_model_{DS}_species')['alpha'], index=SEG75.index)
alpha76 = pd.Series(ALPHA76['alpha'], index=GENES)
shared = [g for g in SEG75.index if g in set(GENES)]
same = [g for g in shared if np.isclose(alpha71[g], alpha76[g], rtol=1e-9, atol=0)]
G4 = {}
for part in PARTS:
    for col in ('T_step', 'delta_S1', 'delta_S2', 'delta_S3', 'se_S1', 'se_S2', 'se_S3'):
        both = [g for g in same if SEG[part].loc[g, 'converged'] and bool(SEG75[part].loc[g, 'converged'])
                and not SEG[part].loc[g, 'separated'] and not bool(SEG75[part].loc[g, 'separated'])]
        a, b = SEG[part].loc[both, col].to_numpy(float), SEG75[part].loc[both, col].to_numpy(float)
        ok = np.isfinite(a) & np.isfinite(b)
        G4[(part, col)] = float(np.max(np.abs(a[ok] - b[ok]) / np.maximum(np.abs(b[ok]), 1.)))
worst = max(G4.values())
assert worst < 1e-6, sorted(G4.items(), key=lambda kv: -kv[1])[:5]
GUARDS.append(f"G4: the U76 segment F-test reproduces 75's T_step, Δ_S and SE on the {len(same):,} of {len(shared):,} shared "
              f"genes with equal α, converged in both runs (max difference relative to max(|value|, 1): {worst:.1e}); "
              f"{len(shared) - len(same)} genes differ in α")
READING.append('G4 compares genes converged and not separated in both segment F-test runs: non-converged genes are '
               "filled by the universe median, which differs between U76 and 71's universe, and separated genes are flagged, "
               'never interpreted. In a development run three separated genes (Cfi, Slc13a4, Kcng1) differed in HC3 SE by up '
               'to 0.6%; T_step and Δ_S agreed to 1e-8.')
DESEQ = pd.read_csv(T75 / 'tables' / f'deseq2_genes_{DS}.csv')
DESEQ_PART = {'condition': 'species', 'relabeling 1': 'swap:IR2A2+Ctrl1A2', 'relabeling 2': 'swap:IR2A2+Ctrl1A4'}
DESEQ_SCOPE = {'S1': 'S1', 'S2': 'S2', 'S3': 'S3', 'PT': 'whole_PT'}


def deseq_stat(part, scope):
    if scope not in DESEQ_SCOPE:
        return None
    block = DESEQ[DESEQ.partition.eq(DESEQ_PART[part]) & DESEQ.scope.eq(DESEQ_SCOPE[scope])].set_index('gene').stat
    return block.dropna()


print(GUARDS[-1])
print(f'Per-gene statistics {time.perf_counter() - t0:.0f} s')

## 5 · Calibration of each statistic on the library sets (protocol 4a, 6)

cameraPR (0.01) of every statistic on U76's 1,495 sets, for the condition and both relabelings; 75's swap rule.
- **Level t:** a call is FDR ≤ 0.05 in either direction; the calibration p is two-sided.
- **F z:** a call is Up; the calibration p is one-sided.

A failing statistic's program p becomes the empirical p (76's rule). Guard: cameraPR on 76's z reproduces its set
tests.

In [ ]:
def calibration_runs(statistics, name, two_sided):
    runs = {part: camera_pr(statistics[part], SETS, inter_gene_cor=CAMERA_COR).set_index('pathway_id') for part in PARTS}
    if two_sided:
        calls = {part: int((runs[part].fdr <= ALPHA).sum()) for part in PARTS}
        null = np.concatenate([runs[p].p.to_numpy(float) for p in PARTS[1:]])
    else:
        calls = {part: int(runs[part].call.sum()) for part in PARTS}
        null = np.concatenate([runs[p].p_one_sided.to_numpy(float) for p in PARTS[1:]])
    verdict = swap_rule(calls['condition'], [calls[p] for p in PARTS[1:]], len(runs['condition']), null)
    row = {'statistic': name, 'tested': len(runs['condition']), **{f'calls {p}': calls[p] for p in PARTS}, **verdict}
    return row, null, runs


t0 = time.perf_counter()
CALIB, NULLS = [], {}
for scope in SCOPES:
    row, NULLS[('level', scope)], _ = calibration_runs({p: LEVEL[(scope, p)].t for p in PARTS}, f'level t · {scope}', True)
    CALIB.append(row)
row, NULLS['z_seg'], runs_seg = calibration_runs(Z_SEG, 'segment F z (U76 refit)', False)
CALIB.append(row)
row, NULLS['z_pos'], runs_pos = calibration_runs(Z_POS, 'positional F z (76)', False)
CALIB.append(row)
for part in PARTS:
    np.testing.assert_allclose(runs_pos[part].p.reindex(TESTS76[part].index), TESTS76[part].p, rtol=1e-8, atol=1e-12)
GUARDS.append("G3: cameraPR on 76's z reproduces its set tests in every partition")
CALIBRATION = pd.DataFrame(CALIB).set_index('statistic')
PASSES = {name: CALIBRATION.loc[name, 'calibration'] == 'pass' for name in CALIBRATION.index}
CALIBRATION.to_csv(tables / 'calibration.csv')
display(CALIBRATION.round(3))
print(f'Calibration {time.perf_counter() - t0:.0f} s')

## 6 · Program tests

- **Replication:** the level t at the claim's scope (PT, S1–S2 or S3), one-sided in the claimed direction (up or down
  in IR).
  - Calibrated when the statistic fails its swap rule.
  - Specificity: the program's own relabeled one-sided p is above 0.05 in both relabelings.
- **Between segments:** the segment F z (Up). **Along the axis:** the positional F z (Up; a sensitivity).
- **Conventional comparator:** the DESeq2 Wald stat at the claim's scope (descriptive; none for S1–S2).

BH runs over the nine confirmatory programs, per question.

In [ ]:
from scipy.stats import false_discovery_control


def bh(p):
    p = pd.Series(p, dtype=float)
    out = pd.Series(np.nan, index=p.index)
    ok = p.notna()
    if ok.any():
        out[ok] = false_discovery_control(p[ok].to_numpy())
    return out


def with_calibration(res, null, passes, two_sided):
    res = res.copy()
    if passes:
        res['p_claimed_cal'], res['p_opposite_cal'] = res.p_claimed, res.p_opposite
        return res
    if two_sided:
        emp = pl.calibrated(res.p.to_numpy(float), null)
        matches = np.isclose(res.p_claimed, res.p / 2)
        res['p_claimed_cal'] = np.where(matches, emp / 2, 1 - emp / 2)
        res['p_opposite_cal'] = np.where(matches, 1 - emp / 2, emp / 2)
    else:
        res['p_claimed_cal'] = pl.calibrated(res.p_claimed.to_numpy(float), null)
        res['p_opposite_cal'] = np.nan
    return res


def level_tests(level, programs, scopes, signs, passes_map=None, nulls=None):
    passes_map = PASSES if passes_map is None else passes_map
    nulls = NULLS if nulls is None else nulls
    rows = []
    for scope in sorted(set(scopes[p] for p in programs)):
        progs = {p: programs[p] for p in programs if scopes[p] == scope}
        stat = {part: level[(scope, part)].t for part in PARTS}
        res = pl.program_test(stat['condition'], progs, {p: signs[p] for p in progs}, inter_gene_cor=CAMERA_COR)
        res = with_calibration(res, nulls[('level', scope)], passes_map[f'level t · {scope}'], True)
        for part in PARTS[1:]:
            res[f'{part} p_claimed'] = pl.program_test(stat[part], progs, {p: signs[p] for p in progs},
                                                       inter_gene_cor=CAMERA_COR).p_claimed
        res['scope'] = scope
        rows.append(res)
    out = pd.concat(rows).reindex(list(programs))
    out['specific'] = (out['relabeling 1 p_claimed'] > ALPHA) & (out['relabeling 2 p_claimed'] > ALPHA)
    return out


def up_tests(statistics, programs, null, passes):
    res = pl.program_test(statistics['condition'], programs, inter_gene_cor=CAMERA_COR)
    res = with_calibration(res, null, passes, False)
    for part in PARTS[1:]:
        res[f'{part} p_up'] = pl.program_test(statistics[part], programs, inter_gene_cor=CAMERA_COR).p_claimed
    return res


def conventional(programs, scopes, signs):
    rows = {}
    for p in programs:
        row = {}
        for part in PARTS:
            stat = deseq_stat(part, scopes[p])
            present = [] if stat is None else [g for g in programs[p] if g in stat.index]
            row[f'DESeq2 {part} p_claimed'] = (np.nan if len(present) < 2 else
                                               float(pl.program_test(stat, {p: present}, {p: signs[p]},
                                                                     inter_gene_cor=CAMERA_COR).p_claimed.iloc[0]))
        rows[p] = row
    return pd.DataFrame(rows).T


def between_tests(programs, adjust=True):
    seg = up_tests(Z_SEG, programs, NULLS['z_seg'], PASSES['segment F z (U76 refit)'])
    pos = up_tests(Z_POS, programs, NULLS['z_pos'], PASSES['positional F z (76)'])
    out = pd.DataFrame({'between p': seg.p_claimed_cal, 'between relabeling 1 p': seg['relabeling 1 p_up'],
                        'between relabeling 2 p': seg['relabeling 2 p_up'], 'axis p': pos.p_claimed_cal,
                        'axis relabeling 1 p': pos['relabeling 1 p_up'], 'axis relabeling 2 p': pos['relabeling 2 p_up']})
    out['between q'] = bh(out['between p']) if adjust else out['between p']
    out['axis q'] = bh(out['axis p']) if adjust else out['axis p']
    return out


I_PROGS = {p: PROGRAMS_ALL[p] for p in CONFIRMATORY}
TEST_I = level_tests(LEVEL, I_PROGS, SCOPE_OF, CLAIM_SIGN)
TEST_I['q_claimed'], TEST_I['q_opposite'] = bh(TEST_I.p_claimed_cal), bh(TEST_I.p_opposite_cal)
TEST_I = TEST_I.join(conventional(I_PROGS, SCOPE_OF, CLAIM_SIGN))
BETWEEN = between_tests(I_PROGS)
display(TEST_I.round(4))
display(BETWEEN.round(4))

## 7 · Segment summaries and the decision rules (protocol 5.1, 5.3)

- **Summary:** P_k is the median over members of the centred IR − control difference in S1, S2 and S3, for the
  condition, the 4 pairs and the 2 relabelings.
- **Signed value:** Q_k = d · P_k, where d is the published direction.
- **Classes:** replication at the claim's scope and localization along S1–S3.
- **Published segment:** it is checked for K-INJ12 (S1–S2) and K-INJ3 (S3).

In [ ]:
def contrasts_of(level, pairs):
    return {'condition': pd.DataFrame({s: level[(s, 'condition')].effect for s in SCOPES}),
            **{part: pd.DataFrame({s: level[(s, part)].effect for s in SCOPES}) for part in PARTS[1:]},
            **{pair: pd.DataFrame({s: pairs[s][pair] for s in SCOPES}) for pair in PAIRS}}


def summarise(contrasts, programs, stat='median'):
    long = pl.segment_summary(contrasts, programs, min_members=MIN_MEMBERS, stat=stat)
    return long.pivot_table(index=['program', 'contrast'], columns='scope', values='value')


def classify(programs, summary, between, test):
    rows = []
    for p in programs:
        d = CLAIM_SIGN[p]
        cond = summary.loc[(p, 'condition'), list(SEGMENTS)].to_numpy(float)
        q_pairs = np.array([summary.loc[(p, pair), list(SEGMENTS)].to_numpy(float) for pair in PAIRS]) * d
        q_relab = np.array([summary.loc[(p, part), list(SEGMENTS)].to_numpy(float) for part in PARTS[1:]]) * d
        loc = pl.localization_class(between.loc[p, 'between q'], cond * d, q_pairs, q_relab)
        scope = SCOPE_OF[p]
        at = summary.loc[(p, 'condition'), scope] * d
        pairs_at = np.array([summary.loc[(p, pair), scope] for pair in PAIRS]) * d
        published = {'S1-S2': ('S1', 'S2'), 'S3': ('S3',)}.get(scope)
        rows.append({
            'program': p, 'd': d, **{f'P {s}': cond[k] for k, s in enumerate(SEGMENTS)}, 'scope': scope,
            'P at scope': at * d, 'pairs with claimed sign': int((pairs_at > 0).sum()),
            'q claimed': test.loc[p, 'q_claimed'], 'q opposite': test.loc[p, 'q_opposite'],
            'specific (relabeled p > 0.05)': bool(test.loc[p, 'specific']),
            'replication': pl.replication_class(test.loc[p, 'q_claimed'], bool(test.loc[p, 'specific']), at, pairs_at,
                                                test.loc[p, 'q_opposite']),
            'localization': loc['class'], 'peak': loc['peak'], 'margin': loc['margin'], 'pairs unanimous': loc['unanimous'],
            'localization specific': loc['specific'], 'relabeled ratio 1': loc['ratios'][0], 'relabeled ratio 2': loc['ratios'][1],
            'present in': ', '.join(loc['present']), 'no difference resolved in': ', '.join(loc['unresolved']),
            'agrees with published segment': (None if published is None else loc['peak'] in published),
            'between q': between.loc[p, 'between q'], 'axis q': between.loc[p, 'axis q'],
            'axis flag': ('changes along the axis but not between segments (descriptive, C14)'
                          if between.loc[p, 'axis q'] <= ALPHA and between.loc[p, 'between q'] > ALPHA else '')})
    out = pd.DataFrame(rows).set_index('program')
    out['informative'] = [pl.informative(r.replication, r.localization) for r in out.itertuples()]
    return out


CONTRASTS = contrasts_of(LEVEL, PAIR_CON)
SUMMARY = summarise(CONTRASTS, I_PROGS)
CLASSES = classify(I_PROGS, SUMMARY, BETWEEN, TEST_I)
display(CLASSES.round(3))

## 8 · Program curves along the axis (coordinate version; display and sensitivity)

Notebook 76's rule:
- centred Δ(x), with the median over usable members, signed by d;
- the 4 IR × control pair curves and the 2 relabelings;
- 76's control segment units.

**Axis agrees** when the floor of the curve's peak equals the label-based peak. Within-segment change is descriptive.

In [ ]:
from pseudospace.aki_along_pt import centre_rows, profile_summary, program_profile


def curve_contrasts(models):
    cond = models['condition']
    ok = cond['stat|converged'].astype(bool) & ~cond['stat|separated'].astype(bool)
    out = {'condition': (centre_rows(cond['delta'], ok)[0], ok)}
    for part in PARTS[1:]:
        m = models[part]
        ok_r = m['stat|converged'].astype(bool) & ~m['stat|separated'].astype(bool)
        out[part] = (centre_rows(m['delta'], ok_r)[0], ok_r)
    for c in CASE:
        for r in REF:
            ok_p = (ok & cond[f'converged|{c}'].astype(bool) & cond[f'converged|{r}'].astype(bool)
                    & ~cond[f'separated|{c}'].astype(bool) & ~cond[f'separated|{r}'].astype(bool))
            diff = cond[f'curve|{c}'] - cond[f'curve|{r}']
            ref_ok = ok_p & np.isfinite(diff).all(axis=1)
            out[f'{c}−{r}'] = (diff - np.nanmedian(diff[ref_ok], axis=0), ok_p)
    return out


CURVE_CON = curve_contrasts(MODELS)


def program_curves(programs, stat='median'):
    curves, rows = {}, []
    for p, members in programs.items():
        d = CLAIM_SIGN[p]
        for name, (centred, ok) in CURVE_CON.items():
            if stat == 'median':
                prof, n = program_profile(centred, GENES, members, usable=ok)
            else:
                idx = pd.Index(GENES).get_indexer(members)
                idx = idx[ok[idx]]
                prof, n = np.nanmean(centred[idx], axis=0) / np.log(2), len(idx)
            curves[(p, name)] = d * prof
            if name == 'condition':
                summ = profile_summary(d * prof, S_GRID)
                rows.append({'program': p, 'members used': n, **summ,
                             'axis peak segment': SEGMENTS[int(np.clip(np.floor(summ['peak']), 0, 2))]})
    return curves, pd.DataFrame(rows).set_index('program')


CURVES, CURVE_SUMMARY = program_curves(I_PROGS)
CLASSES['axis peak'] = CURVE_SUMMARY['axis peak segment']
CLASSES['axis'] = np.where(CLASSES['axis peak'] == CLASSES.peak, 'agrees', 'differs')
display(CURVE_SUMMARY.round(3))

## 9 · Sensitivities (protocol 7)

- **Mean instead of median** (O4).
- **Untrimmed units:** every exported PT unit (15,013).
- **The c19 arm:** notebook 76's arm (ii), pseudobulk only. Its 80 injured-PT units, labelled cTAL by the package,
  are added at their neighbour-placed PT label (73 S3, 7 S2), with counts from the run folder. No model refit.
- **Segment model check:** the peak of the median of the refit's Δ_S against the pseudobulk peak.

The segment F-test q stays the analysed units' in every arm (it is a fitted statistic).

In [ ]:
t0 = time.perf_counter()
SENS = pd.DataFrame(index=CLASSES.index)
SUMMARY_MEAN = summarise(CONTRASTS, I_PROGS, stat='mean')
CLASSES_MEAN = classify(I_PROGS, SUMMARY_MEAN, BETWEEN, TEST_I)
SENS['mean: replication'], SENS['mean: localization'] = CLASSES_MEAN.replication, CLASSES_MEAN.localization

# Library rule of aki_inputs: the measured one-to-one panel (any count in the kept units) without the mt genes
kept = UNITS.obs_names.get_indexer(INP['obs'].index)
panel_idx = UNITS.var_names.get_indexer(PANEL)
panel_counts_kept = UNITS.X[kept][:, panel_idx]
measured = np.asarray(panel_counts_kept.sum(axis=0)).ravel() > 0
lib_genes = [g for g, m in zip(PANEL, measured) if m and g not in set(MT)]
lib_idx = UNITS.var_names.get_indexer(lib_genes)
gene_idx = UNITS.var_names.get_indexer(GENES)
all_library = np.asarray(UNITS.X[:, lib_idx].sum(axis=1)).ravel()
np.testing.assert_allclose(all_library[kept], LIBRARY)
del panel_counts_kept


def arm(counts, library, specimen, segment):
    rates = pseudobulks(counts, library, specimen, segment)
    level, pairs = level_statistics(rates, np.log(np.asarray(counts.mean(axis=0)).ravel() + 1e-300))
    nulls, passes = {}, {}
    for scope in SCOPES:
        row, nulls[('level', scope)], _ = calibration_runs({p: level[(scope, p)].t for p in PARTS}, f'level t · {scope}', True)
        passes[f'level t · {scope}'] = row['calibration'] == 'pass'
    test = level_tests(level, I_PROGS, SCOPE_OF, CLAIM_SIGN, passes_map=passes, nulls=nulls)
    test['q_claimed'], test['q_opposite'] = bh(test.p_claimed_cal), bh(test.p_opposite_cal)
    summary = summarise(contrasts_of(level, pairs), I_PROGS)
    return classify(I_PROGS, summary, BETWEEN, test), summary


all_counts = UNITS.X[:, gene_idx].tocsr()
all_specimen = UNITS.obs['sample'].astype(str).to_numpy()
all_segment = UNITS.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
CLASSES_ALL, SUMMARY_ALL = arm(all_counts, all_library, all_specimen, all_segment)
SENS['untrimmed: replication'], SENS['untrimmed: localization'] = CLASSES_ALL.replication, CLASSES_ALL.localization

PLACEMENT = pd.read_csv(T76 / 'tables' / 'placement_units.csv', index_col=[0, 1])
PLACED = PLACEMENT.loc['c19 onto PT'].copy()
PLACED['sample'] = PLACEMENT.loc['c19', 'sample'].reindex(PLACED.index)          # the onto-PT rows carry no sample
assert len(PLACED) == 80 and set(PLACED.majority) <= {'PT-S1', 'PT-S2', 'PT-S3'} and PLACED['sample'].notna().all()
with h5py.File(run_aki / '02_qc' / 'units_qc.h5ad', 'r') as f:
    qc_index = pd.Index(read_elem(f['obs'][f['obs'].attrs['_index']]).astype(str))
    qc_var = pd.Index(read_elem(f['var'][f['var'].attrs['_index']]).astype(str))
    qc_rows = qc_index.get_indexer(PLACED.index.astype(str))
    assert (qc_rows >= 0).all() and qc_var.equals(UNITS.var_names)
    QC = read_elem(f['layers/counts']).tocsr()[qc_rows]
c19_counts = QC[:, gene_idx]
c19_library = np.asarray(QC[:, lib_idx].sum(axis=1)).ravel()
c19_specimen = PLACED['sample'].astype(str).to_numpy()
c19_segment = PLACED.majority.str.replace('PT-', '', regex=False).to_numpy()
CLASSES_C19, SUMMARY_C19 = arm(sparse.vstack([COUNTS, c19_counts]).tocsr(), np.r_[LIBRARY, c19_library],
                               np.r_[SPECIMEN, c19_specimen], np.r_[SEGMENT, c19_segment])
SENS['c19 arm: replication'], SENS['c19 arm: localization'] = CLASSES_C19.replication, CLASSES_C19.localization
SENS['c19 arm: P S3'] = CLASSES_C19['P S3']
c19_table = pd.crosstab(c19_specimen, c19_segment)
display(c19_table)

delta = pd.DataFrame({s: SEG['condition'][f'delta_{s}'] for s in SEGMENTS}).astype(float)
ok_seg = SEG['condition'].converged & ~SEG['condition'].separated
delta = (delta - delta[ok_seg].median()) / np.log(2)
for p in CLASSES.index:
    members = [g for g in I_PROGS[p] if ok_seg.get(g, False)]
    vals = delta.loc[members].median().to_numpy() * CLASSES.loc[p, 'd']
    SENS.loc[p, 'refit Δ_S peak'] = SEGMENTS[int(np.argmax(vals))]
SENS['refit Δ_S peak agrees'] = SENS['refit Δ_S peak'] == CLASSES.peak
display(SENS)
print(f'Sensitivities {time.perf_counter() - t0:.0f} s')

## 10 · Secondary programs (same analyses, nominal p, labelled secondary)

In [ ]:
SEC_PROGS = {p: PROGRAMS_ALL[p] for p in SECONDARY}
SEC_TEST = level_tests(LEVEL, SEC_PROGS, SCOPE_OF, CLAIM_SIGN)
SEC_TEST['q_claimed'], SEC_TEST['q_opposite'] = SEC_TEST.p_claimed_cal, SEC_TEST.p_opposite_cal     # nominal
SEC_BETWEEN = between_tests(SEC_PROGS, adjust=False)
SEC_SUMMARY = summarise(CONTRASTS, SEC_PROGS)
SEC_CLASSES = classify(SEC_PROGS, SEC_SUMMARY, SEC_BETWEEN, SEC_TEST)
SEC_CLASSES['p kind'] = 'nominal (secondary)'
SEC_CURVES, SEC_CURVE_SUMMARY = program_curves(SEC_PROGS)
SEC_CLASSES['axis peak'] = SEC_CURVE_SUMMARY['axis peak segment']
SEC_CLASSES['axis'] = np.where(SEC_CLASSES['axis peak'] == SEC_CLASSES.peak, 'agrees', 'differs')
display(SEC_CLASSES[['scope', 'replication', 'localization', 'peak', 'agrees with published segment', 'between q']].round(3))

## 11 · Added information (protocol 4c; one templated line per program)

In [ ]:
def lines_for(classes):
    out = {}
    for p, r in classes.iterrows():
        flags = ['label limitation (c15 dropped, c19 labelled cTAL: injured units under-represented, most in S3)']
        if r.get('agrees with published segment') is not None and not pd.isna(r.get('agrees with published segment')):
            flags.append('peak ' + ('in' if r['agrees with published segment'] else 'outside') + ' the published segment')
        if p in KIRITA_TIME:
            flags.append(f'Kirita observed this state: {KIRITA_TIME[p]} (time point of our IRI unknown)')
        row = {'source': str(META.loc[p, 'source']).split(' (')[0], 'claim': META.loc[p, 'published_claim'].split(' (')[0].strip(),
               'replication': r.replication, 'scope': r.scope, 'value': r['P at scope'],
               'pairs': int(r['pairs with claimed sign']), 'q': r['q claimed'], 'localization': r.localization,
               'peak': r.peak, 'unresolved': [s for s in str(r['no difference resolved in']).split(', ') if s],
               'axis': r.axis, 'flags': flags}
        out[p] = pl.added_information(row)
    return pd.Series(out, name='added information')


ADDED = pd.concat([lines_for(CLASSES), lines_for(SEC_CLASSES)])
for p, line in ADDED.items():
    print(f'{p}: {line}')
replicated_states = [p for p in KIRITA_TIME if CLASSES.loc[p, 'replication'] == 'replicated']
TIME_LINE = ('Kirita states replicated here: ' + (', '.join(f'{p} ({KIRITA_TIME[p]})' for p in replicated_states) or 'none')
             + '. Read only as "compatible with the states Kirita observed at these times", never as a time-point claim.')
print(TIME_LINE)

## 12 · Figures

- **Small multiples** (protocol 4d), one row per program:
  1. the program curve: the condition thick, the 4 IR × control pairs thin, the relabelings dashed;
  2. the member heatmap trio: control mean, IR mean and their difference;
  3. segment bars.
- **Summary heatmap:** programs × S1/S2/S3, coloured by Q = d · P; dots count the agreeing pairs; the box marks the
  localized segment and the line the published segment.

In [ ]:
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle

ALL_RATES, BIN_MAJORITY, BIN_NAMES = pl.bin_rates(COUNTS, LIBRARY, POSITION, SPECIMEN, SEGMENT, np.arange(len(GENES)),
                                                  n_bins=N_BINS, floor=FLOOR)
spec_index = {n: i for i, n in enumerate(BIN_NAMES)}
ref_mean = np.mean([ALL_RATES[spec_index[s]] for s in REF], axis=0)
case_mean = np.mean([ALL_RATES[spec_index[s]] for s in CASE], axis=0)
diff_bins = case_mean - ref_mean
diff_bins = diff_bins - np.median(diff_bins, axis=0)
gene_pos = {g: i for i, g in enumerate(GENES)}
SUMMARY_FOR = {p: pd.concat([SUMMARY, SEC_SUMMARY]).xs(p, level='program') for p in {**I_PROGS, **SEC_PROGS}}
ALL_CURVES = {**CURVES, **SEC_CURVES}
PUBLISHED = {'S1': [0], 'S2': [1], 'S3': [2], 'S1-S2': [0, 1], 'PT': []}


def heat_rows(members):
    rows = members[:MAX_ROWS]
    idx = [gene_pos[g] for g in rows]
    order = np.argsort(np.argmax(ref_mean[:, idx], axis=0), kind='stable')
    return [rows[i] for i in order], [idx[i] for i in order]


def family_figure(programs, classes, title, stem):
    n = len(programs)
    fig, axes = plt.subplots(n, 5, figsize=(183 * MM, 34 * MM * n), gridspec_kw={'width_ratios': [1.25, .7, .7, .7, .75]},
                             squeeze=False)
    for i, p in enumerate(programs):
        r = classes.loc[p]
        ax = axes[i, 0]
        for part, style in (('relabeling 1', '--'), ('relabeling 2', ':')):
            ax.plot(S_GRID, ALL_CURVES[(p, part)], color=RELABEL_COLOR, lw=.7, ls=style)
        for pair in PAIRS:
            ax.plot(S_GRID, ALL_CURVES[(p, pair)], color=IR_COLOR, lw=.5, alpha=.6)
        ax.plot(S_GRID, ALL_CURVES[(p, 'condition')], color=INK, lw=1.4)
        for b in (1, 2):
            ax.axvline(b, color=LIGHT, lw=.6, zorder=0)
        ax.axhline(0, color=MUTED, lw=.5, zorder=0)
        ax.set_xlim(S_GRID.min(), S_GRID.max())
        ax.set_ylabel(f'd·Δ (log2), d = {int(r.d):+d}')
        ax.set_title(f'{p} · {META.loc[p, "source"].split(" (")[0]} · {META.loc[p, "direction"]}', loc='left')
        if i == n - 1:
            ax.set_xlabel('segment units (control cuts; within-segment shape descriptive)')
        rows, idx = heat_rows(PROGRAMS_ALL[p])
        both = np.concatenate([ref_mean[:, idx], case_mean[:, idx]], axis=0)
        centred = both - both.mean(axis=0)
        lim = np.nanpercentile(np.abs(centred), 98) or 1
        for j, (mat, lab) in enumerate(((centred[:N_BINS], 'control'), (centred[N_BINS:], 'IR'), (diff_bins[:, idx], 'IR − control'))):
            ax = axes[i, j + 1]
            vmax = lim if j < 2 else max(np.nanpercentile(np.abs(diff_bins[:, idx]), 98), .1)
            ax.imshow(mat.T, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax, interpolation='nearest')
            ax.set_yticks(range(len(rows)))
            ax.set_yticklabels(rows if j == 0 else [], fontsize=3.2)
            ax.set_xticks([])
            group = REF if j == 0 else CASE if j == 1 else REF
            maj = BIN_MAJORITY.loc[list(group)].mode().iloc[0]
            for b in range(N_BINS):
                ax.add_patch(Rectangle((b - .5, len(rows) - .5), 1, max(len(rows) * .04, .6), clip_on=False,
                                       color={'S1': '#cfd8dc', 'S2': '#90a4ae', 'S3': '#455a64'}[maj[b]], lw=0))
            ax.set_title(f'{lab} ({len(rows)} of {len(PROGRAMS_ALL[p])})', fontsize=5.5, fontweight='normal')
        ax = axes[i, 4]
        x = np.arange(3)
        ax.bar(x, [SUMMARY_FOR[p].loc['condition', s] for s in SEGMENTS], color=[LIGHT] * 3, edgecolor=INK, lw=.5)
        for k, s in enumerate(SEGMENTS):
            ax.scatter([k] * 4, [SUMMARY_FOR[p].loc[pair, s] for pair in PAIRS], s=4, color=IR_COLOR, zorder=3)
            ax.scatter([k] * 2, [SUMMARY_FOR[p].loc[part, s] for part in PARTS[1:]], s=8, marker='x', color=RELABEL_COLOR, zorder=3)
        for k in PUBLISHED.get(SCOPE_OF[p], []):
            ax.add_patch(Rectangle((k - .45, ax.get_ylim()[0]), .9, np.ptp(ax.get_ylim()), fill=False, ec=IR_COLOR, lw=.8, ls='--'))
        ax.axhline(0, color=MUTED, lw=.5)
        ax.set_xticks(x)
        ax.set_xticklabels(SEGMENTS)
        ax.set_ylabel('IR − control (log2)')
        ax.set_title(f'{r.replication} at {r.scope}\n{r.localization}', fontsize=5.5, fontweight='normal')
    fig.suptitle(title, x=0, ha='left', fontweight='bold')
    fig.tight_layout()
    save(fig, stem)


family_figure(CONFIRMATORY, CLASSES, 'Injury family I: published injured-PT programs (IR − control)', 'fig80a_injury_programs')


def summary_heatmap(classes, summary, stem, title):
    progs = list(classes.index)
    q = np.array([[summary.loc[(p, 'condition'), s] * classes.loc[p, 'd'] for s in SEGMENTS] for p in progs])
    fig, ax = plt.subplots(figsize=(150 * MM, (22 + 5.5 * len(progs)) * MM))
    im = ax.imshow(np.clip(q, -2, 2), cmap='RdBu_r', norm=TwoSlopeNorm(0, -2, 2), aspect='auto')
    for i, p in enumerate(progs):
        r = classes.loc[p]
        for k, s in enumerate(SEGMENTS):
            agree = sum(np.sign(summary.loc[(p, pair), s] * r.d) == np.sign(q[i, k]) for pair in PAIRS)
            ax.text(k, i, '•' * int(agree), ha='center', va='center', fontsize=5, color=INK)
        if str(r.localization).startswith('localized'):
            k = SEGMENTS.index(r.peak)
            ax.add_patch(Rectangle((k - .5, i - .5), 1, 1, fill=False, ec=INK, lw=1.4))
        for k in PUBLISHED.get(SCOPE_OF[p], []):
            ax.plot([k - .4, k + .4], [i + .42, i + .42], color=IR_COLOR, lw=1.2)
        ax.text(2.6, i, f'{r.replication} · {r.localization}', va='center', fontsize=5.5)
    ax.set_xticks(range(3))
    ax.set_xticklabels(SEGMENTS)
    ax.set_yticks(range(len(progs)))
    ax.set_yticklabels([f'{p} ({META.loc[p, "direction"]})' for p in progs])
    ax.set_xlim(-.5, 2.5)
    fig.colorbar(im, ax=ax, orientation='horizontal', fraction=.04, pad=.12, aspect=30,
                 label='Q = d · median centred IR − control (log2); d = published direction')
    ax.set_title(title, loc='left')
    save(fig, stem)


summary_heatmap(CLASSES, SUMMARY, 'fig80b_summary_heatmap', 'Published injury programs × segment (dots: pairs agreeing; box: localized; line: published segment)')
summary_heatmap(SEC_CLASSES, SEC_SUMMARY, 'supp80_secondary_heatmap', 'Secondary injury programs (nominal p)')
family_figure(SECONDARY, SEC_CLASSES, 'Secondary injury programs (nominal p)', 'supp80_secondary_programs')

## 13 · Outputs

In [ ]:
META.to_csv(tables / 'programs.csv')
MEMBERS_TABLE.to_csv(tables / 'programs_members.csv', index=False)
pd.concat({s: RATES[s] for s in SCOPES}, axis=1).to_csv(tables / 'pseudobulk_rates.csv.gz')
gene_rows = []
for p, members in {**I_PROGS, **SEC_PROGS}.items():
    for g in members:
        row = {'program': p, 'gene': g, 'z_seg': Z_SEG['condition'][g], 'z_pos': Z_POS['condition'][g],
               'converged (76)': bool(STAT76['condition'].converged[g]), 'separated (76)': bool(STAT76['condition'].separated[g])}
        for s in SCOPES:
            row[f'condition {s}'] = LEVEL[(s, 'condition')].effect[g]
            row[f't {s}'] = LEVEL[(s, 'condition')].t[g]
            for pair in PAIRS:
                row[f'{pair} {s}'] = PAIR_CON[s][pair][g]
        gene_rows.append(row)
GENE_LEVEL = pd.DataFrame(gene_rows).merge(MEMBERS_TABLE.rename(columns={'program_id': 'program'}), on=['program', 'gene'], how='left')
GENE_LEVEL.to_csv(tables / 'gene_level.csv', index=False)
share = []
for p in I_PROGS:
    d, scope = CLAIM_SIGN[p], SCOPE_OF[p]
    vals = CONTRASTS['condition'].loc[I_PROGS[p], scope] * d
    pairs_ok = np.all([CONTRASTS[pair].loc[I_PROGS[p], scope] * d > 0 for pair in PAIRS], axis=0)
    base = CONTRASTS['condition'][scope] * d
    base_pairs = np.all([CONTRASTS[pair][scope] * d > 0 for pair in PAIRS], axis=0)
    share.append({'program': p, 'share claimed sign': float((vals > 0).mean()),
                  'share claimed sign, all 4 pairs': float(pairs_ok.mean()), 'universe share': float((base > 0).mean()),
                  'universe share, all 4 pairs': float(base_pairs.mean())})
GENE_SHARES = pd.DataFrame(share).set_index('program')
display(GENE_SHARES.round(3))
PROGRAM_TESTS = TEST_I.join(BETWEEN)
PROGRAM_TESTS.to_csv(tables / 'program_tests.csv')
SEC_TEST.join(SEC_BETWEEN).to_csv(tables / 'program_tests_secondary.csv')
pd.concat({'median': SUMMARY, 'mean': SUMMARY_MEAN, 'untrimmed (median)': SUMMARY_ALL, 'c19 arm (median)': SUMMARY_C19},
          names=['statistic']).to_csv(tables / 'segment_summary.csv')
np.savez_compressed(tables / 'profiles.npz', grid=GRID, s=S_GRID, **{f'{p}|{name}': v for (p, name), v in ALL_CURVES.items()})
pd.concat([CURVE_SUMMARY, SEC_CURVE_SUMMARY]).to_csv(tables / 'profile_summaries.csv')
CLASSES.join(GENE_SHARES).to_csv(tables / 'classes.csv')
SEC_CLASSES.to_csv(tables / 'classes_secondary.csv')
ADDED.to_csv(tables / 'added_information.csv')
SENS.to_csv(tables / 'sensitivity.csv')
CLASSES_MEAN.to_csv(tables / 'sensitivity_mean_classes.csv')
CLASSES_ALL.to_csv(tables / 'sensitivity_untrimmed_classes.csv')
CLASSES_C19.to_csv(tables / 'sensitivity_c19_classes.csv')

## 14 · Readout (protocol 5.4) and the abandonment rule

In [ ]:
display(CLASSES.informative.value_counts())
resolved = CLASSES.localization.str.startswith('localized') | CLASSES.localization.isin(['graded', 'uniform within resolution'])
n_resolved = int(resolved.sum())
ABANDON = n_resolved < 3
print(f'Injury family: {n_resolved} of 9 programs reach a resolved localization class (abandonment rule: fewer than 3) → '
      + ('position adds no resolvable information to the injury claims' if ABANDON else 'the rule is not met'))
outside = [p for p in CLASSES.index if CLASSES.loc[p, 'agrees with published segment'] is not None
           and not pd.isna(CLASSES.loc[p, 'agrees with published segment']) and not bool(CLASSES.loc[p, 'agrees with published segment'])]
print('Programs with a published segment whose peak lies outside it:', outside or 'none')
print(TIME_LINE)
display(CLASSES[['scope', 'replication', 'localization', 'peak', 'agrees with published segment', 'axis', 'informative']])

## 15 · How the protocol was read

In [ ]:
READING.extend([
    'Programs are read from the committed docs/results/known-programs/ files (relative to the repository), with sha256 asserted.',
    "Relabeled specificity of a replication test uses the program's relabeled one-sided p in the claimed direction, as "
    'the protocol writes; the relabeled p are nominal.',
    'The S1–S2 scope (K-INJ12, Dixon PTS1-2) is the pseudobulk of S1 and S2 units together; 75 has no DESeq2 scope '
    'for it, so its conventional comparator is NaN.',
    'The c19 arm adds the 80 c19 units at the majority PT label of their 30 nearest PT units (76\'s placement table); '
    'their library follows aki_inputs\' rule (the measured one-to-one panel without mt genes).',
    'The untrimmed and c19 arms keep the segment F-test q of the analysed units and recompute every pseudobulk '
    'quantity, the level t and its calibration.',
    'Heatmap rows are the first 40 members in the paper\'s ranking order (data-independent), ordered by the control peak bin.',
    'The pair curves are centred by the pair\'s own median gene at each grid point over genes finite there.',
    'Kirita state times are quoted from the paper (protocol 1.5) for wording only.',
])
HOW = pd.Series(READING, name='how the protocol was read')
HOW.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
for line in READING:
    print('·', line)

## 16 · Run record

In [ ]:
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'protocol': 'docs/results/pt-known-programs.md (commit b90a682)',
          'pass1_fingerprint': FINGERPRINT, 'program_sha256': PROGRAM_SHA, 'units': int(len(POSITION)), 'genes': len(GENES),
          'library_sets': len(SETS), 'segment_cuts': [T12, T23], 'guards': GUARDS,
          'calibration': CALIBRATION.reset_index().to_dict('records'),
          'abandonment_rule': {'injury resolved': n_resolved, 'met': bool(ABANDON)}, 'time_line': TIME_LINE,
          'n_jobs': N_JOBS, 'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(record, indent=1, default=str))
print(f"Done in {record['seconds']:.0f} s")